In [1]:
!pip -q install -U keras-nlp sacrebleu rouge-score nltk

In [2]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd

import keras
import keras_nlp

import sacrebleu

2026-08-20 16:52:32.112993: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787244752.134333     131 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787244752.140978     131 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787244752.158290     131 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787244752.158310     131 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787244752.158313     131 computation_placer.cc:177] computation placer alr

In [3]:
def find_file(filename: str, search_roots=("/kaggle/input", "/kaggle/working", ".")) -> str:
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.rglob(filename):
            return str(p)
    raise FileNotFoundError(f"Could not find {filename} under {search_roots}")

def find_dir_contains(hint: str, root="/kaggle/input") -> str:
    hint = hint.lower()
    for d in os.listdir(root):
        if hint in d.lower().replace("_", "-"):
            return os.path.join(root, d)
    raise FileNotFoundError(f"Could not find dataset folder containing '{hint}' in {root}. Found: {os.listdir(root)}")

In [4]:
import re
import unicodedata

KRIYA_MUL_PATH = find_file("kriya_mul_list.txt")
KRIYA_VIBOKTI_PATH = find_file("kriya_vibokti_list.txt")
EXCEPTIONS_PATH = find_file("exceptional_cases.txt")

def load_lines(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return [line.strip() for line in f if line.strip()]

def _nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s)

def _variants_bn(s: str):
    s = _nfc(s)
    out = {s}
    out.add(s.replace("য়", "য়"))
    out.add(s.replace("য়", "য়"))
    out.add(s.replace("িয়া", "িয়"))
    out.add(s.replace("িয়", "িয়া"))
    return out

kriya_mul_list = [_nfc(x) for x in load_lines(KRIYA_MUL_PATH)]

vib_pairs = []
for line in load_lines(KRIYA_VIBOKTI_PATH):
    parts = line.split()
    if len(parts) >= 2:
        vib_pairs.append((_nfc(parts[0]), _nfc(parts[1])))

exc_pairs = []
for line in load_lines(EXCEPTIONS_PATH):
    parts = line.split()
    if len(parts) >= 2:
        exc_pairs.append((_nfc(parts[0]), _nfc(parts[1])))

sadhu_to_cholito_vibokti = {}
for sfx_s, sfx_c in vib_pairs:
    for v in _variants_bn(sfx_s):
        sadhu_to_cholito_vibokti[v] = _nfc(sfx_c)

exceptions_map = {}
for s, c in exc_pairs:
    for v in _variants_bn(s):
        exceptions_map[v] = _nfc(c)

EXTRA_VIBOKTI = {
    "িয়াছি": "েছি",
    "িয়াছি": "েছি",
    "য়াছি": "েছি",
    "য়াছি": "েছি",
    "ইয়াছি": "েছি",
    "ইয়াছি": "েছি",
    "িয়াছে": "েছে",
    "িয়াছে": "েছে",
    "য়াছে": "েছে",
    "য়াছে": "েছে",
    "িয়াছেন": "েছেন",
    "িয়াছেন": "েছেন",
    "য়াছেন": "েছেন",
    "য়াছেন": "েছেন",
}
for k, v in EXTRA_VIBOKTI.items():
    sadhu_to_cholito_vibokti.setdefault(_nfc(k), _nfc(v))

kriya_mul_list = sorted({r for r in kriya_mul_list if r}, key=len, reverse=True)
sadhu_suffixes = sorted(sadhu_to_cholito_vibokti.keys(), key=len, reverse=True)

PUNCT_CHARS = ".,;:!?\"'“”‘’।,，…()[]{}<>"
PUNCT_CLASS = re.escape(PUNCT_CHARS)
_tok_re = re.compile(rf"^([{PUNCT_CLASS}]*)?(.*?)([{PUNCT_CLASS}]*)?$")

def split_punct(tok: str):
    m = _tok_re.match(tok)
    if not m:
        return "", tok, ""
    return (m.group(1) or ""), (m.group(2) or ""), (m.group(3) or "")

def sadhu_to_cholito(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return text

    parts = re.split(r"(\s+)", text)
    out = []

    for part in parts:
        if not part or part.isspace():
            out.append(part)
            continue

        lead, core, trail = split_punct(part)
        core = _nfc(core)

        if not core:
            out.append(part)
            continue

        if core in exceptions_map:
            out.append(f"{lead}{exceptions_map[core]}{trail}")
            continue

        replaced = None

        for root in kriya_mul_list:
            if root in core and core != root:
                suffix = core.split(root, 1)[1]
                mapped = sadhu_to_cholito_vibokti.get(suffix)
                if mapped is not None:
                    replaced = root + mapped
                    break

        if replaced is None:
            for sadhu_sfx in sadhu_suffixes:
                if sadhu_sfx and core.endswith(sadhu_sfx) and len(core) > len(sadhu_sfx):
                    stem = core[:-len(sadhu_sfx)]
                    replaced = stem + sadhu_to_cholito_vibokti[sadhu_sfx]
                    break

        out.append(f"{lead}{replaced}{trail}" if replaced else part)

    return "".join(out)

print("Loaded rule files:")
print("kriya_mul_list:", KRIYA_MUL_PATH)
print("kriya_vibokti_list:", KRIYA_VIBOKTI_PATH)
print("exceptional_cases:", EXCEPTIONS_PATH)
print("Quick test:", sadhu_to_cholito("আমি করিয়াছি।"))

Loaded rule files:
kriya_mul_list: /kaggle/input/rules-for-sadhu-to-cholit/kriya_mul_list.txt
kriya_vibokti_list: /kaggle/input/rules-for-sadhu-to-cholit/kriya_vibokti_list.txt
exceptional_cases: /kaggle/input/rules-for-sadhu-to-cholit/exceptional_cases.txt
Quick test: আমি করেছি।


In [5]:
POETRY_DIR = find_dir_contains("free-bengali-poetry")  

txt_files = sorted([str(p) for p in Path(POETRY_DIR).glob("*.txt")])

print("POETRY_DIR:", POETRY_DIR)
print("TXT files:", [Path(x).name for x in txt_files])
SADHU_DIR = find_dir_contains("datasets")
print("SADHU_DIR:", SADHU_DIR)
LINES_PATH = None
for root, dirs, files in os.walk(SADHU_DIR):
    for file in files:
        if file.endswith(".txt"):
            LINES_PATH = os.path.join(root, file)
            break
    if LINES_PATH:
        break
print("LINES_PATH:", LINES_PATH)
with open(LINES_PATH, "r", encoding="utf-8", errors="ignore") as f:
    new_lines = [line.strip() for line in f if line.strip()]
print("New lines loaded:", len(new_lines))
print("Sample new line:", new_lines[0][:100])

POETRY_DIR: /kaggle/input/free-bengali-poetry
TXT files: ['ishwarchandragupta.txt', 'jatindramohan.txt', 'jibanananda.txt', 'kaminiroy.txt', 'madhusudan.txt', 'nazrulislam.txt', 'rabindranath.txt', 'satyendranath.txt', 'sukanta.txt', 'sukumar.txt']
SADHU_DIR: /kaggle/input/datasets
LINES_PATH: /kaggle/input/datasets/dnandi/dataset-of-sadhu-bhasha-sentences/test_dataset.txt
New lines loaded: 18373
Sample new line: উত্তপ্ত হয়ে ওঠে নি


In [6]:
def load_poem_blocks_from_txt(txt_path: str):
    text = Path(txt_path).read_text(encoding="utf-8", errors="ignore")
    blocks = [b.strip() for b in text.split("\n\n") if b.strip()]
    blocks = [b for b in blocks if len(b) >= 40]
    return blocks

def poem_to_lines(block: str):
    lines = [x.strip() for x in block.splitlines()]
    lines = [x for x in lines if x and len(x) >= 4]
    return lines

# --- Load poetry ---
poem_blocks = []
for fp in txt_files:
    poem_blocks.extend(load_poem_blocks_from_txt(fp))

poem_blocks = list(dict.fromkeys(poem_blocks))

all_lines = []
for b in poem_blocks:
    all_lines.extend(poem_to_lines(b))

all_lines = list(dict.fromkeys(all_lines))

print("Poetry blocks:", len(poem_blocks))
print("Poetry lines :", len(all_lines))
print("Sample poetry line:", all_lines[0] if all_lines else "None")

# Filter out very short lines and lines that are just numbers/punctuation
def is_valid_sentence(s: str) -> bool:
    # Must have at least 5 Bengali characters
    bengali_chars = sum(1 for c in s if '\u0980' <= c <= '\u09FF')
    return bengali_chars >= 5

new_lines = [s for s in new_lines if is_valid_sentence(s)]
new_lines = list(dict.fromkeys(new_lines))  # Remove duplicates

print("Filtered new lines:", len(new_lines))
print("Sample new line:", new_lines[0][:100] if new_lines else "None")

# --- Combine all lines ---
all_lines = list(dict.fromkeys(all_lines + new_lines))
print("Total combined lines:", len(all_lines))

Poetry blocks: 2709
Poetry lines : 79311
Sample poetry line: বল দেখি এ জগতে ধার্মিক কে হয়,
Filtered new lines: 18168
Sample new line: উত্তপ্ত হয়ে ওঠে নি
Total combined lines: 97479


In [7]:
import numpy as np

pairs = []
changed_count = 0
unchanged_count = 0

for s in all_lines:
    s = s.strip()
    if not s:
        continue

    out = sadhu_to_cholito(s)
    if isinstance(out, str):
        out = out.strip()
        if out and out != s:
            pairs.append((s, out))
            changed_count += 1
        elif out and out == s:
            unchanged_count += 1

print("Total pairs (changed only):", len(pairs))
print("Unchanged lines (skipped):", unchanged_count)
if pairs:
    print("Example:\nIN :", pairs[0][0], "\nOUT:", pairs[0][1])

INSTRUCTION = "সাধু ভাষা থেকে চলিত ভাষায় রূপান্তর কর। শুধু চলিত বাক্য দাও।"

TEMPLATE = (
    "Instruction:\n{instruction}\n\n"
    "Input:\n{input}\n\n"
    "Output:\n{output}"
)

rng = np.random.default_rng(42)
idx = rng.permutation(len(pairs))

n_train = int(0.90 * len(pairs))
n_val = int(0.05 * len(pairs))

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:]

train_texts = [
    TEMPLATE.format(instruction=INSTRUCTION, input=pairs[i][0], output=pairs[i][1])
    for i in train_idx
]
val_texts = [
    TEMPLATE.format(instruction=INSTRUCTION, input=pairs[i][0], output=pairs[i][1])
    for i in val_idx
]

test_inputs = [pairs[i][0] for i in test_idx]
test_refs = [pairs[i][1] for i in test_idx]

print("Train/Val/Test:", len(train_texts), len(val_texts), len(test_inputs))
if train_texts:
    print("Prompt sample:\n", train_texts[0][:400], "...")

Total pairs (changed only): 2143
Unchanged lines (skipped): 95336
Example:
IN : মনে হ’ল গিয়াছে বালাই; 
OUT: মনে হ’ল গেছে বালাই;
Train/Val/Test: 1928 107 108
Prompt sample:
 Instruction:
সাধু ভাষা থেকে চলিত ভাষায় রূপান্তর কর। শুধু চলিত বাক্য দাও।

Input:
যে সুখের তরে পাপে ধর্ম ভাবিয়াছে

Output:
যে সুখের তরে পাপে ধর্ম ভাবেছে ...


In [8]:
def split_into_chunks(text: str, max_len: int = 50) -> list:
    """Split a sentence into smaller chunks at punctuation."""
    import re
    chunks = re.split(r'[।,.;\n]+', text)
    chunks = [c.strip() for c in chunks if len(c.strip()) >= 10]
    return chunks

augmented_pairs = []
for inp, out in pairs[:]:
    inp_chunks = split_into_chunks(inp)
    out_chunks = split_into_chunks(out)
    if len(inp_chunks) == len(out_chunks) and len(inp_chunks) > 1:
        for i_c, o_c in zip(inp_chunks, out_chunks):
            if i_c and o_c and i_c != o_c:
                augmented_pairs.append((i_c, o_c))

print("Augmented pairs:", len(augmented_pairs))
pairs.extend(augmented_pairs[:1000])
pairs = list(dict.fromkeys(pairs))
print("Total pairs after augmentation:", len(pairs))

Augmented pairs: 992
Total pairs after augmentation: 3128


In [9]:
import tensorflow as tf
import keras
import keras_nlp
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")

gemma2lm = keras_nlp.models.GemmaCausalLM.from_preset(
    "gemma2_2b_en",
    dtype="float16",
)

gemma2lm.preprocessor.sequence_length = 128 

LORA_RANK = 4  
gemma2lm.backbone.enable_lora(rank=LORA_RANK)

optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma2lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
    jit_compile=False, 
)

gemma2lm.summary()

I0000 00:00:1787244761.643460     131 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787244761.649275     131 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:431: UserWarning: `build()` was called on layer 'gemma_causal_lm', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,617,270,528 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,617,270,528 (4.88 GB)

 Trainable params: 2,928,640 (11.17 MB)

 Non-trainable params: 2,614,341,888 (4.87 GB)

In [10]:
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

class BestLoraCallback(keras.callbacks.Callback):
    """Save LoRA weights only when val_loss improves."""
    def __init__(self, filepath, monitor='val_loss', mode='min'):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.best = float('inf') if mode == 'min' else -float('inf')
        
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        if self.mode == 'min' and current < self.best:
            self.best = current
            self.model.backbone.save_lora_weights(self.filepath)
            print(f"✅ Best model saved (val_loss = {current:.4f})")
        elif self.mode == 'max' and current > self.best:
            self.best = current
            self.model.backbone.save_lora_weights(self.filepath)
            print(f"✅ Best model saved (val_loss = {current:.4f})")

# Use a fixed learning rate; ReduceLROnPlateau will adjust it
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma2lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
    jit_compile=False,
)

# Callbacks
callbacks = [
    BestLoraCallback(
        "/kaggle/working/gemma2_sadhu2cholito_best_lora.lora.h5",
        monitor='val_loss'
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    ),
]

history = gemma2lm.fit(
    train_texts,
    validation_data=(val_texts,),
    epochs=25,
    batch_size=1,
    callbacks=callbacks,
)

Epoch 1/25


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1928/1928 ━━━━━━━━━━━━━━━━━━━━ 0s 258ms/step - loss: 1.0206 - sparse_categorical_accuracy: 0.8828✅ Best model saved (val_loss = 0.9514)
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 592s 279ms/step - loss: 1.0206 - sparse_categorical_accuracy: 0.8828 - val_loss: 0.9514 - val_sparse_categorical_accuracy: 0.8860 - learning_rate: 5.0000e-05
Epoch 2/25
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - loss: 0.8464 - sparse_categorical_accuracy: 0.8974✅ Best model saved (val_loss = 0.9199)
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 523s 271ms/step - loss: 0.8464 - sparse_categorical_accuracy: 0.8974 - val_loss: 0.9199 - val_sparse_categorical_accuracy: 0.8857 - learning_rate: 5.0000e-05
Epoch 3/25
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - loss: 0.8050 - sparse_categorical_accuracy: 0.9022✅ Best model saved (val_loss = 0.9028)
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 523s 271ms/step - loss: 0.8050 - sparse_categorical_accuracy: 0.9022 - val_loss: 0.9028 - val_sparse_categorical_accuracy: 0.8889 - learning_rate: 5.0000e-05
Epo

In [11]:
!pip -q install -U evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [12]:
import os
import numpy as np
import sacrebleu
import nltk
from nltk.translate.meteor_score import meteor_score
import evaluate  # Hugging Face metrics library

nltk.download('wordnet', quiet=True)

# --- LOAD BEST LORA WEIGHTS (if saved) ---
best_lora_path = "/kaggle/working/gemma2_sadhu2cholito_best_lora.lora.h5"
if os.path.exists(best_lora_path):
    gemma2lm.backbone.load_lora_weights(best_lora_path)
    print("✅ Loaded best LoRA weights from", best_lora_path)
else:
    print("⚠️ Best LoRA file not found. Using current model (untrained?).")

# --- Setup sampler and generation ---
sampler = keras_nlp.samplers.TopKSampler(k=5, seed=22)
gemma2lm.compile(sampler=sampler)

def model_convert(s: str) -> str:
    prompt = TEMPLATE.format(instruction=INSTRUCTION, input=s, output="")
    raw = gemma2lm.generate(prompt, max_length=512)
    # Extract the output after "Output:" if present, otherwise return the raw text
    if "Output:" in raw:
        out = raw.split("Output:", 1)[1].strip()
    else:
        out = raw.strip()
    return out

# --- Evaluate on test set ---
EVAL_N = min(200, len(test_inputs))
preds = []
refs = test_refs[:EVAL_N]

print("Generating predictions...")
for i, inp in enumerate(test_inputs[:EVAL_N]):
    pred = model_convert(inp)
    preds.append(pred)
    # Print first 5 examples for debugging
    if i < 5:
        print(f"\n--- Sample {i} ---")
        print(f"INPUT : {inp[:100]}...")
        print(f"PRED  : {pred[:100]}...")
        print(f"REF   : {refs[i][:100]}...")
        # Print the raw generated text for the first example to verify extraction
        if i == 0:
            raw_example = gemma2lm.generate(TEMPLATE.format(instruction=INSTRUCTION, input=inp, output=""), max_length=512)
            print(f"\n[DEBUG] Raw generated text:\n{raw_example[:300]}...")

# --- 1. BLEU & chrF (sacrebleu) ---
bleu = sacrebleu.corpus_bleu(preds, [refs])
chrf = sacrebleu.corpus_chrf(preds, [refs])
print(f"\nBLEU : {bleu.score:.2f}")
print(f"chrF : {chrf.score:.2f}")

# --- 2. METEOR (nltk) ---
meteor_scores = []
for p, r in zip(preds, refs):
    if p.strip() and r.strip():
        meteor_scores.append(meteor_score([r.split()], p.split()))
if meteor_scores:
    print(f"METEOR   : {np.mean(meteor_scores):.4f}")
else:
    print("METEOR: no valid pairs.")

# --- 3. ROUGE (Hugging Face evaluate) ---
rouge = evaluate.load('rouge')
rouge_results = rouge.compute(predictions=preds, references=refs, use_stemmer=False)
print(f"ROUGE-1 F1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2 F1: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L F1: {rouge_results['rougeL']:.4f}")

# --- 4. Additional word-level BLEU (optional) ---
bleu_word = sacrebleu.corpus_bleu(preds, [refs], tokenize='none')
print(f"BLEU (word): {bleu_word.score:.2f}")

# --- Print a few full examples for sanity ---
print("\n" + "="*50)
print("Full examples:")
for i in range(min(3, len(preds))):
    print(f"\n--- Example {i} ---")
    print(f"IN  : {test_inputs[i]}")
    print(f"REF : {refs[i]}")
    print(f"PRED: {preds[i]}")

[nltk_data] Error loading wordnet: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


✅ Loaded best LoRA weights from /kaggle/working/gemma2_sadhu2cholito_best_lora.lora.h5
Generating predictions...


I0000 00:00:1787252172.506994     131 service.cc:152] XLA service 0x230e2c50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787252172.507050     131 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787252172.507056     131 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787252178.268709     131 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787252190.377008     131 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



--- Sample 0 ---
INPUT : না, চুরি বন্ধ হইয়াছে...
PRED  : না, চুরি বন্ধ হইেছে...
REF   : না, চুরি বন্ধ হইেছে...

[DEBUG] Raw generated text:
Instruction:
সাধু ভাষা থেকে চলিত ভাষায় রূপান্তর কর। শুধু চলিত বাক্য দাও।

Input:
না, চুরি বন্ধ হইয়াছে

Output:
না, চুরি বন্ধ হইেছে...

--- Sample 1 ---
INPUT : এক স্থানে বাম দিকে একটি রক্ত বাহির হইয়াছে, তাহা দিয়া একটু ুক্্র পথ আছে...
PRED  : এক স্থানে বাম দিকে একটি রক্ত বাহির হইেছে, তাহা দিয়া একটু ুক্্র পথ...
REF   : এক স্থানে বাম দিকে একটি রক্ত বাহির হইেছে, তাহা দিয়া একটু ুক্্র পথ আছে...

--- Sample 2 ---
INPUT : লুণ্ঠিত শিথিল বাহু, পড়িয়াছে খসি...
PRED  : লুণ্ঠিত শিথিল বাহু, পড়েছে খসি...
REF   : লুণ্ঠিত শিথিল বাহু, পড়েছে খসি...

--- Sample 3 ---
INPUT : বুঝিয়াছি; শুপুরীর সারিগুলো দিনরাত হাওয়ায় যে উঠিতেছে নড়ে,দিনরাত কথা নয়, ক্ষীরের মতন ফুল বুকে ধরে, তাদ...
PRED  : বুঝিয়াছি; শ...
REF   : বুঝেছি; শুপুরীর সারিগুলো দিনরাত হাওয়ায় যে উঠিতেছে নড়ে,দিনরাত কথা নয়, ক্ষীরের মতন ফুল বুকে ধরে, তাদের...

--- Sample 4 ---
INPUT : আজ বিবাহ সময়ে উপযুক্ত

ROUGE-1 F1: 0.0093
ROUGE-2 F1: 0.0000
ROUGE-L F1: 0.0093
BLEU (word): 43.37

Full examples:

--- Example 0 ---
IN  : না, চুরি বন্ধ হইয়াছে
REF : না, চুরি বন্ধ হইেছে
PRED: না, চুরি বন্ধ হইেছে

--- Example 1 ---
IN  : এক স্থানে বাম দিকে একটি রক্ত বাহির হইয়াছে, তাহা দিয়া একটু ুক্্র পথ আছে
REF : এক স্থানে বাম দিকে একটি রক্ত বাহির হইেছে, তাহা দিয়া একটু ুক্্র পথ আছে
PRED: এক স্থানে বাম দিকে একটি রক্ত বাহির হইেছে, তাহা দিয়া একটু ুক্্র পথ

--- Example 2 ---
IN  : লুণ্ঠিত শিথিল বাহু, পড়িয়াছে খসি
REF : লুণ্ঠিত শিথিল বাহু, পড়েছে খসি
PRED: লুণ্ঠিত শিথিল বাহু, পড়েছে খসি


In [13]:
def convert_interactive():
    print("Type a sentence (blank to exit).")
    while True:
        s = input("\nSadhu> ").strip()
        if not s:
            break
        print("Cholito> ", model_convert(s))

convert_interactive()

Type a sentence (blank to exit).



Sadhu>  ভাবিলাম তোমাকে দেখিলে বোধ হয় কিছু শ্রান্তি দূর হইবে, পিপাসাও নিবারণ হইবে


Cholito>  ভাবিলাম তোমাকে দেখিলে বোধ হয় কিছু শ্রান্তি দূর হইবে, পিপাসাও



Sadhu>  
